# **Cell 1: Instalasi & Mount**

In [1]:
!pip install -q segment-anything torch torchvision opencv-python ultralytics
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 491.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
Mounted at /content/drive


# **Cell 2: Import & Download SAM**

In [2]:
import os, cv2, torch, numpy as np, random
from segment_anything import sam_model_registry, SamPredictor
import matplotlib.pyplot as plt

# Download SAM ViT-B jika belum ada
sam_checkpoint = '/content/sam_vit_b_01ec64.pth'
if not os.path.exists(sam_checkpoint):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {sam_checkpoint}
model_type = "vit_b"
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)
print("✅ SAM loaded")

✅ SAM loaded


# **Cell 3: Fungsi Anotasi SAM**

In [3]:
def get_sam_bbox_grid(image):
    h, w = image.shape[:2]
    predictor.set_image(image)
    points = []
    for i in range(3):
        for j in range(3):
            points.append([int(w * (j+1) / 4), int(h * (i+1) / 4)])
    point_coords = np.array(points)
    point_labels = np.ones(len(points))
    masks, scores, _ = predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        multimask_output=False
    )
    if len(scores) == 0:
        return None
    best_idx = np.argmax(scores)
    mask = masks[best_idx]
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    x, y, bw, bh = x_min, y_min, x_max - x_min, y_max - y_min
    if (bw * bh) < (h * w) * 0.02:
        return None
    return (x, y, bw, bh)

# **Cell 4: Proses Dataset Positif**

In [ ]:
base = '/content/drive/MyDrive/BlockchainAI/detector_dataset'
fallback_count = 0
total_pos = 0
for split in ['train', 'val']:
    img_dir = os.path.join(base, 'images', split)
    lbl_dir = os.path.join(base, 'labels', split)
    for fname in os.listdir(img_dir):
        if fname.startswith('lorem'): continue  # negatif
        img_path = os.path.join(img_dir, fname)
        img = cv2.imread(img_path)
        if img is None: continue
        total_pos += 1
        rect = get_sam_bbox_grid(img)

        if rect is None:
            fallback_count += 1
            # fallback full-frame
            h, w = img.shape[:2]
            rect = (0, 0, w, h)
        x, y, bw, bh = rect
        dw, dh = 1./img.shape[1], 1./img.shape[0]
        xc = (x + bw/2) * dw
        yc = (y + bh/2) * dh
        nw = bw * dw
        nh = bh * dh
        base_name = os.path.splitext(fname)[0]
        with open(os.path.join(lbl_dir, base_name + '.txt'), 'w') as f:
            f.write(f'0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}\n')
    print(f"{split}: selesai.")
fallback_pct = (fallback_count / total_pos) * 100 if total_pos else 0
print(f"Total positif: {total_pos}, Fallback: {fallback_count} ({fallback_pct:.2f}%)")
if fallback_pct > 5:
    print("⚠️ Fallback > 5%, hentikan dan evaluasi!")
    # Hentikan eksekusi atau beri peringatan keras
    assert False, "Fallback terlalu tinggi, perlu review prompt"

# **Cell 5: Visualisasi Sampel (20 gambar)**

In [ ]:
samples = random.sample([f for f in os.listdir(os.path.join(base, 'images', 'train')) if not f.startswith('lorem')], min(20, total_pos))
for fname in samples:
    img_path = os.path.join(base, 'images', 'train', fname)
    img = cv2.imread(img_path)
    lbl_path = os.path.join(base, 'labels', 'train', os.path.splitext(fname)[0]+'.txt')
    with open(lbl_path) as f:
        line = f.readline().split()
        if line:
            xc, yc, nw, nh = map(float, line[1:])
            h, w = img.shape[:2]
            x1 = int((xc - nw/2)*w)
            y1 = int((yc - nh/2)*h)
            x2 = int((xc + nw/2)*w)
            y2 = int((yc + nh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(fname)
    plt.axis('off')
    plt.show()

# **Cell 6: Hapus cache & Training**

In [ ]:
!rm -f /content/drive/MyDrive/BlockchainAI/detector_dataset/labels/train/*.cache
!rm -f /content/drive/MyDrive/BlockchainAI/detector_dataset/labels/val/*.cache

from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/drive/MyDrive/BlockchainAI/detector_dataset/data.yaml',
    epochs=100, batch=16, imgsz=640, patience=15,
    name='phase2b_sam', exist_ok=True,
    optimizer='AdamW', lr0=0.001
)

# **Cell 7: Evaluasi dengan Phase 2A**

In [ ]:
import pandas as pd
gt = pd.read_csv('/content/drive/MyDrive/BlockchainAI/demo_test_ground_truth.csv')
gt_clean = gt[gt['category'].isin(['obat','non_obat'])]
obat_files = gt_clean[gt_clean['category']=='obat']['filename'].tolist()
non_obat_files = gt_clean[gt_clean['category']=='non_obat']['filename'].tolist()
demo_dir = '/content/drive/MyDrive/BlockchainAI/demo_test'
detected_obat = 0
failed_obat = []
for fname in obat_files:
    res = model(os.path.join(demo_dir, fname), conf=0.5, verbose=False)
    if len(res[0].boxes) > 0: detected_obat += 1
    else: failed_obat.append(fname)
false_pos = 0
fp_files = []
for fname in non_obat_files:
    res = model(os.path.join(demo_dir, fname), conf=0.5, verbose=False)
    if len(res[0].boxes) > 0:
        false_pos += 1
        fp_files.append(fname)
recall = detected_obat / len(obat_files) if obat_files else 0
print(f"Recall: {recall:.2%}, FP: {false_pos}")
# Simpan model
import shutil
model.save('/content/drive/MyDrive/BlockchainAI/models/detector_model.pt')
model.export(format='tflite', imgsz=640)
shutil.move('runs/detect/phase2b_sam/weights/best.tflite', '/content/drive/MyDrive/BlockchainAI/models/detector_model.tflite')